# Build Graph from Sumo Traci with sidewalks

In [11]:
import numpy as np
import networkx as nx
import traci

In [12]:
def build_graph():
    G = nx.DiGraph()

    # Get all edges (skip internal)
    edge_ids = [e for e in traci.edge.getIDList() if not e.startswith(":")]

    for eid in edge_ids:
        from_node = traci.edge.getFromJunction(eid)
        to_node = traci.edge.getToJunction(eid)

        lid = f"{eid}_0"  # Assume single lane per edge for simplicity
        length = traci.lane.getLength(lid)
        speed = traci.edge.getLastStepMeanSpeed(eid)

        # Add nodes (junctions)
        fx, fy = traci.junction.getPosition(from_node)
        tx, ty = traci.junction.getPosition(to_node)
        G.add_node(from_node, x=fx, y=fy)
        G.add_node(to_node, x=tx, y=ty)

        # Add directed edge with attributes
        G.add_edge(
            from_node,
            to_node,
            id=eid,
            length=length,
            speed=speed,
            weight=length / speed if speed > 0 else float("inf")
        )

    all_shortest_path_lengths = dict(nx.all_pairs_shortest_path_length(G))
    return G, all_shortest_path_lengths


In [13]:
Sumo_config = [
    'sumo',
    '-c', "data/grid/grid.sumocfg",
    '--step-length', '1',
    '--start'
]
traci.start(Sumo_config)

# Maze State
# G, all_shortest_path_lengths = build_graph() # Rebuild graph with new maze

 Retrying in 1 seconds


(22, 'SUMO 1.24.0')

In [ ]:
# Get all pedestrian-only sidewalk lanes
sidewalk_lanes = []

for lane_id in traci.lane.getIDList():
    
    if 'pedestrian' not in traci.lane.getAllowed(lane_id):
        continue
    
    sidewalk_lanes.append(lane_id)

In [18]:
crossing_edge = ":A2_c0"
tmp = traci.lane.getLinks(crossing_edge + "_0")

In [19]:
tmp

((':A2_w0_0', True, True, False, '', 'M', 's', 8.399999999999977),)

In [10]:
traci.close()

FatalTraCIError: Not connected.